In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('Incident_Management_CSV.csv', sep=';')
print(df.head())

In [ ]:
df.count()

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M')
print("\n--- Timestamp Conversion Verified ---")

# 3. Sort the data to ensure events are in chronological order per Case ID (The Sequence)
df = df.sort_values(by=['Case ID', 'Timestamp'])
df.head()

In [ ]:
print("\n--- Dataset Summary ---")
print(f"Total Records Loaded: {len(df):,}")
print(f"Unique Cases (Incidents): {df['Case ID'].nunique():,}")
print(df.dtypes)

In [ ]:
df.head()

In [ ]:
df.columns = df.columns.str.lower()
df.columns

df.rename(columns={'case id': 'case_id'}, inplace=True)
df.columns

#df.to_csv('Incident_Management_CSV_cleaned.csv', index=False)

In [ ]:
#loading to postgres database
import os
from sqlalchemy import URL, create_engine

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ.get("PGUSER", "postgres"),
    password=os.environ.get("PGPASSWORD", "0000"),
    host=os.environ.get("PGHOST", "localhost"),
    port=int(os.environ.get("PGPORT", "5432")),
    database="Incident_Management",
)

engine = create_engine(url)

df.to_sql(
    "incident_data",
    engine,
    if_exists="replace",
    index=False,
    chunksize=5000,
    method="multi",
)

print(f"Uploaded {len(df):,} rows to Incident_Management.incident_data")

In [ ]:
df.head()

In [ ]:
def load_data(path="Incident_Management_CSV_cleaned.csv"):
    df = pd.read_csv(path)

    # Standardize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    return df

def clean_data(df):
    df = df.copy()
    df = df.dropna(axis=1, how="all")
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )
    string_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for col in string_columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
        )

        # Explicitly mark missing values
        df[col] = df[col].fillna("Unknown")

    numeric_columns = df.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_columns:
        df[col] = df[col].fillna(
            df[col].median()
        )
    if "resolver" in df.columns:
        df["resolver_missing"] = (
            df["resolver"] == "Unknown"
        ).astype(int)

    return df


def create_incident_dataset(df):
    """
    Convert event-level process data into one row per incident.
    """

    df = df.copy()

    df = df.sort_values(
        ["case_id", "timestamp"]
    )

    incidents = []

    for case_id, group in df.groupby("case_id"):

        group = group.sort_values("timestamp")

        first = group.iloc[0]

        start_time = group["timestamp"].min()
        end_time = group["timestamp"].max()

        duration_hours = (
            end_time - start_time
        ).total_seconds() / 3600

        incidents.append({
            "case_id": case_id,

            "variant": first.get(
                "variant", "Unknown"
            ),

            "priority": first.get(
                "priority", "Unknown"
            ),

            "reporter": first.get(
                "reporter", "Unknown"
            ),

            "issue_type": first.get(
                "issue_type", "Unknown"
            ),

            "report_channel": first.get(
                "report_channel", "Unknown"
            ),

            "short_description": first.get(
                "short_description", "Unknown"
            ),

            "customer_satisfaction": first.get(
                "customer_satisfaction",
                np.nan
            ),

            "event_count": len(group),

            "unique_events": group["event"].nunique(),

            "unique_resolvers": group["resolver"].nunique(),

            "start_time": start_time,

            "end_time": end_time,

            "duration_hours": duration_hours,

            "escalated": int(
                group["event"]
                .str.contains(
                    "escalat",
                    case=False,
                    na=False
                )
                .any()
            ),

            "level_2_involvement": int(
                group["event"]
                .str.contains(
                    "level 2",
                    case=False,
                    na=False
                )
                .any()
            )
        })

    incident_df = pd.DataFrame(incidents)

    return incident_df


def finalize_numeric_missing_values(df):

    df = df.copy()

    numeric_columns = df.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_columns:

        if df[col].isna().any():

            median = df[col].median()

            if pd.isna(median):
                median = 0

            df[col] = df[col].fillna(median)

    return df


if __name__ == "__main__":

    df = load_data()

    print("Original shape:", df.shape)

    df = clean_data(df)

    print("Event-level shape:", df.shape)

    incident_df = create_incident_dataset(df)

    incident_df = finalize_numeric_missing_values(
        incident_df
    )

    print(
        "Incident-level shape:",
        incident_df.shape
    )

    print("\nMissing values:")
    print(
        incident_df.isna().sum()
    )

    incident_df.to_csv(
        "incidents_clean.csv",
        index=False
    )

    print(
        "\nSaved: data/incidents_clean.csv"
    )